# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-10-31 17:47:54--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-10-31 17:47:54--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip’

enron_spam_data.zip 100%[===================>]  14.92M  48.7MB/s    in 0.3s    

2026-10-31 17:47:55 (48.7 MB

In [2]:
import pandas as pd

In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

0.5092834262664611

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [16]:
import json
from sklearn.feature_extraction.text import CountVectorizer

message_ids = enron_spam_data['Message ID'].astype(int)
test_mask = message_ids.mod(30).eq(0)
train_data = enron_spam_data.loc[~test_mask].reset_index(drop=True)
test_data = enron_spam_data.loc[test_mask].reset_index(drop=True)

subject_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 1),
    min_df=3,
    max_features=3000
)
message_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 1),
    min_df=3,
    max_features=10000
)

train_subject_matrix = subject_vectorizer.fit_transform(
    train_data['Subject'].fillna('').astype(str)
)
train_message_matrix = message_vectorizer.fit_transform(
    train_data['Message'].fillna('').astype(str)
)
test_subject_matrix = subject_vectorizer.transform(
    test_data['Subject'].fillna('').astype(str)
)
test_message_matrix = message_vectorizer.transform(
    test_data['Message'].fillna('').astype(str)
)

subject_feature_names = subject_vectorizer.get_feature_names_out()
message_feature_names = message_vectorizer.get_feature_names_out()

print(f'Training emails: {len(train_data)}')
print(f'Test emails: {len(test_data)}')
print(f'Subject unigram features: {len(subject_feature_names)}')
print(f'Message unigram features: {len(message_feature_names)}')

Training emails: 32592
Test emails: 1124
Subject unigram features: 3000
Message unigram features: 10000


Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [ ]:
def sparse_row_to_features(subject_matrix, message_matrix, row_index):
    subject_row = subject_matrix.getrow(row_index)
    message_row = message_matrix.getrow(row_index)
    features = {
        f's:{subject_feature_names[column]}': int(count)
        for column, count in zip(subject_row.indices, subject_row.data)
    }
    features.update({
        f'm:{message_feature_names[column]}': int(count)
        for column, count in zip(message_row.indices, message_row.data)
    })
    return json.dumps(features, separators=(',', ':'))

train_features = pd.DataFrame({
    'Message ID': train_data['Message ID'].astype(int),
    'features_json': [
        sparse_row_to_features(train_subject_matrix, train_message_matrix, row_index)
        for row_index in range(len(train_data))
    ]
})
test_features = pd.DataFrame({
    'Message ID': test_data['Message ID'].astype(int),
    'features_json': [
        sparse_row_to_features(test_subject_matrix, test_message_matrix, row_index)
        for row_index in range(len(test_data))
    ]
})

train_features.to_csv('train-features.tsv', sep='\t', index=False)
test_features.to_csv('test-features.tsv', sep='\t', index=False)

print(f'Wrote {len(train_features)} training rows and {len(test_features)} test rows')
train_features.head()

Wrote 32592 training rows and 1124 test rows


,Message ID,features_json
0,1,"{""subject:resources"":1,""subject:inc"":1,""messag..."
1,2,"{""subject:calpine"":1,""subject:daily"":1,""subjec..."
2,3,"{""subject:re"":1,""subject:issue"":1,""message:fro..."
3,4,"{""subject:meter"":1,""subject:nov"":1,""subject:al..."
4,5,"{""subject:gas"":1,""subject:mcmullen"":1,""subject..."


Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


In [ ]:
import json
from collections import Counter

train_features = pd.read_csv('train-features.tsv', sep='\t')
labels_by_id = enron_spam_data.set_index('Message ID')['Spam/Ham']
class_feature_counts = {'ham': Counter(), 'spam': Counter()}

for message_id, features_json in zip(
    train_features['Message ID'],
    train_features['features_json']
):
    label = labels_by_id.loc[int(message_id)]
    class_feature_counts[label].update(json.loads(features_json))

vocabulary = sorted(
    set(class_feature_counts['ham']) | set(class_feature_counts['spam'])
)
vocabulary_size = len(vocabulary)
alpha = 1
ham_total = sum(class_feature_counts['ham'].values())
spam_total = sum(class_feature_counts['spam'].values())

feature_probabilities = pd.DataFrame({
    'feature': vocabulary,
    'ham_probability': [
        (class_feature_counts['ham'][feature] + alpha) /
        (ham_total + alpha * vocabulary_size)
        for feature in vocabulary
    ],
    'spam_probability': [
        (class_feature_counts['spam'][feature] + alpha) /
        (spam_total + alpha * vocabulary_size)
        for feature in vocabulary
    ]
})

print(f'Vocabulary size: {vocabulary_size}')
print(f'Ham feature count: {ham_total}')
print(f'Spam feature count: {spam_total}')
feature_probabilities.head()

Vocabulary size: 40000
Ham feature count: 5796110
Spam feature count: 4319435


,feature,ham_probability,spam_probability
0,message:00,1.084798e-03,0.001330
1,message:00 00,1.528415e-04,0.000006
2,message:00 01,7.025227e-06,0.000003
3,message:00 010,1.713470e-07,0.000014
4,message:00 10,1.730605e-05,0.000007


Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [8]:
feature_probabilities.to_csv(
    'feature-probabilities.tsv',
    sep='\t',
    index=False
)
feature_probabilities.head()

,feature,ham_probability,spam_probability
0,message:00,1.084798e-03,0.001330
1,message:00 00,1.528415e-04,0.000006
2,message:00 01,7.025227e-06,0.000003
3,message:00 010,1.713470e-07,0.000014
4,message:00 10,1.730605e-05,0.000007


Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

In [10]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

train_features = pd.read_csv('train-features.tsv', sep='\t')
feature_probabilities = pd.read_csv('feature-probabilities.tsv', sep='\t')
feature_names = feature_probabilities['feature'].tolist()
feature_to_column = {feature: column for column, feature in enumerate(feature_names)}

train_labels = labels_by_id.reindex(train_features['Message ID']).to_numpy()
class_priors = {
    'ham': np.mean(train_labels == 'ham'),
    'spam': np.mean(train_labels == 'spam')
}
log_ham_prior = np.log(class_priors['ham'])
log_spam_prior = np.log(class_priors['spam'])
log_ham_features = np.log(feature_probabilities['ham_probability'].to_numpy())
log_spam_features = np.log(feature_probabilities['spam_probability'].to_numpy())

def make_feature_matrix(feature_frame):
    row_indices = []
    column_indices = []
    values = []
    for row_index, features_json in enumerate(feature_frame['features_json']):
        feature_counts = json.loads(features_json)
        for feature, count in feature_counts.items():
            column = feature_to_column.get(feature)
            if column is not None:
                row_indices.append(row_index)
                column_indices.append(column)
                values.append(count)
    return csr_matrix(
        (values, (row_indices, column_indices)),
        shape=(len(feature_frame), len(feature_names)),
        dtype=float
    )

def predict_nb_probabilities(feature_frame):
    feature_matrix = make_feature_matrix(feature_frame)
    ham_log_scores = feature_matrix @ log_ham_features + log_ham_prior
    spam_log_scores = feature_matrix @ log_spam_features + log_spam_prior
    log_normalizers = np.logaddexp(ham_log_scores, spam_log_scores)
    return pd.DataFrame({
        'Message ID': feature_frame['Message ID'].astype(int).to_numpy(),
        'ham': np.exp(ham_log_scores - log_normalizers),
        'spam': np.exp(spam_log_scores - log_normalizers)
    })

train_predictions = predict_nb_probabilities(train_features)
train_predictions.head()

,Message ID,ham,spam
0,1,1.0,0.000000e+00
1,2,1.0,3.154737e-24
2,3,1.0,0.000000e+00
3,4,1.0,3.822867e-298
4,5,1.0,2.001360e-54


Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [11]:
train_predictions.to_csv(
    'train-predictions.tsv',
    sep='\t',
    index=False
)
train_predictions.head()

,Message ID,ham,spam
0,1,1.0,0.000000e+00
1,2,1.0,3.154737e-24
2,3,1.0,0.000000e+00
3,4,1.0,3.822867e-298
4,5,1.0,2.001360e-54


Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.
If the test data has tokens that you did not model, just ignore them.

In [12]:
test_features = pd.read_csv('test-features.tsv', sep='\t')
test_predictions = predict_nb_probabilities(test_features)
test_predictions.head()

,Message ID,ham,spam
0,0,0.186706,8.132942e-01
1,30,1.000000,2.785096e-176
2,60,1.000000,5.068461e-18
3,90,1.000000,6.067124e-49
4,120,1.000000,0.000000e+00


Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [13]:
test_predictions.to_csv(
    'test-predictions.tsv',
    sep='\t',
    index=False
)
test_predictions.head()

,Message ID,ham,spam
0,0,0.186706,8.132942e-01
1,30,1.000000,2.785096e-176
2,60,1.000000,5.068461e-18
3,90,1.000000,6.067124e-49
4,120,1.000000,0.000000e+00


Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

In [14]:
test_labels = labels_by_id.reindex(
    test_predictions['Message ID']
).to_numpy()
true_spam = test_labels == 'spam'
predicted_spam_probability = test_predictions['spam'].to_numpy()

thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
positive_count = np.sum(true_spam)
negative_count = np.sum(~true_spam)
roc_rows = []

for threshold in thresholds:
    predicted_spam = predicted_spam_probability >= threshold
    true_positive_rate = np.sum(predicted_spam & true_spam) / positive_count
    false_positive_rate = np.sum(predicted_spam & ~true_spam) / negative_count
    roc_rows.append((threshold, false_positive_rate, true_positive_rate))

roc_data = pd.DataFrame(
    roc_rows,
    columns=['threshold', 'false_positive_rate', 'true_positive_rate']
)
roc_data.head()

,threshold,false_positive_rate,true_positive_rate
0,0.01,0.032609,0.993007
1,0.02,0.032609,0.993007
2,0.03,0.030797,0.993007
3,0.04,0.028986,0.993007
4,0.05,0.028986,0.993007


Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [15]:
roc_data.to_csv('roc.tsv', sep='\t', index=False)
roc_data.head()

,threshold,false_positive_rate,true_positive_rate
0,0.01,0.032609,0.993007
1,0.02,0.032609,0.993007
2,0.03,0.030797,0.993007
3,0.04,0.028986,0.993007
4,0.05,0.028986,0.993007


Submit "roc.tsv" in Gradescope.

## Part 7: OpenAI API Key

Sign up for an OpenAI API acount at https://platform.openai.com/.
If you already have an OpenAI account (e.g. for ChatGPT), you may reuse that account.

Create a "dx704" project at https://platform.openai.com/settings/organization/projects, setup a $10/month spend limit, and create an API key for the project at https://platform.openai.com/api-keys.
You will use this API key for projects in later weeks.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

### Acknowledgement

No other people provided assistance. I used pandas and NumPy for dataset handling and calculations, scikit-learn's `CountVectorizer` for extracting word-count features, and SciPy sparse matrices for efficient feature storage and prediction. GitHub Copilot Chat in Visual Studio Code assisted with interpreting the assignment, implementing and debugging the Naive Bayes calculations, and checking the output files.

[Copilot Chat transcript](file:///Users/christopherabbas/Library/Application%20Support/Code/User/workspaceStorage/d690c5667068afdbf94ff162bc48cae2/GitHub.copilot-chat/transcripts/e7d90d9c-3572-4733-8668-e31c0026bd5e.jsonl)